## Project Summary and Next Steps

### What we completed
- Confirmed the competition structure: two targets (`Smoking` and `Gamma_GT`), `ID` as identifier, and no target leakage into features.
- Validated the train/test schema and submission template before modeling.
- Built a meaning-based feature cleaning pipeline:
  - handled blindness flags for `Eyesight_L`/`Eyesight_R`
  - reconstructed missing `LDL` where allowed
  - added missingness indicators and panel-level flags
  - excluded assay identifiers (`Urine_Plate`, `POC_Batch`) from the baseline model features
- Created train-only imputation rules and preserved missingness as signal rather than zero-filling.
- Applied `RobustScaler` only to continuous numeric features for models that need scaled inputs.
- Trained separate baseline models for each task:
  - `CatBoostClassifier` for `Smoking`
  - `CatBoostRegressor` for `Gamma_GT` using `log1p(Gamma_GT)`
- Compared CatBoost vs. XGBoost on the same validation split and found CatBoost to be the stronger initial baseline.
- Refit the selected CatBoost models on the full training data and generated a competition-ready `submission.csv`.
- Ran a 5-fold stability check to compare pooled out-of-fold performance and assess consistency across folds.

### Key takeaways
- The modeling workflow was built around leakage-safe preprocessing.
- The target `Gamma_GT` was handled with log-transform regression to respect the positive, skewed distribution.
- CatBoost was selected as the current leader for both tasks, with XGBoost kept as a strong comparison baseline.
- The submission file is generated and ready to be updated whenever model changes are made.

### Recommended next steps
1. Review the 5-fold results and decide whether CatBoost remains the final choice.
2. Tune CatBoost hyperparameters more deliberately for both targets.
3. Test targeted feature engineering, especially around lab-derived features and missingness signals.
4. Compare a few additional models (for example LightGBM or a tuned XGBoost variant) on the same folds.
5. Rebuild `submission.csv` after any final model changes and keep a clean record of the best-performing run.
6. If the leaderboard score is weak, focus on feature quality and validation discipline before adding too much complexity.

> Current status: the project is in the improvement/tuning stage, with a validated baseline and a generated submission pipeline already in place.

# SMOKESCREEN Modeling Project

## Objective
Build two separate prediction models for the competition:

- **`Smoking` classification:** predict a smoker probability in `[0, 1]`; evaluated with AUC.
- **`Gamma_GT` regression:** predict a positive Gamma-GT value in U/L; evaluated with RMSLE. Because the target is right-skewed, train the regressor on `log1p(Gamma_GT)` and invert predictions with `expm1`.

The test file has features and `ID`, but no target labels. Keep `ID` out of model features, and never use either target as a predictor.

## Machine-Learning Lifecycle
1. Problem framing and input validation (completed)
2. Exploratory data analysis and data-quality decisions (initial cleaning complete)
3. Validation design and leakage-safe preprocessing (five-fold comparison complete)
4. Train separate classification and regression baselines (CatBoost currently leads XGBoost on five-fold metrics)
5. Improve features and tune each model separately (**current stage**)
6. Refit selected models on all labeled training data and generate a validated submission (`submission.csv` available; regenerate after model changes)

Five-fold metrics are a stronger stability check than one holdout but are not a completely untouched final estimate because iteration counts were selected from an earlier holdout. The test file has no labels, so final predictions cannot be scored locally.

## Step 1: Problem Framing and Input Validation

The competition has two independent targets and therefore needs two model families: a classifier that produces `Smoking` probabilities, and a regressor for `Gamma_GT`. In this first step we confirm the files and target/feature boundaries before exploring or fitting anything.

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path.cwd()
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

TARGETS = ["Smoking", "Gamma_GT"]
ID_COLUMN = "ID"
FEATURES = [column for column in train.columns if column not in TARGETS + [ID_COLUMN]]

In [11]:
# Confirm targets, feature boundaries, and submission template before modeling.
assert all(target in train.columns for target in TARGETS)
assert all(target not in test.columns for target in TARGETS)
assert ID_COLUMN in train.columns and ID_COLUMN in test.columns
assert set(FEATURES) == set(test.columns) - {ID_COLUMN}
assert train["Smoking"].dropna().isin([0, 1]).all()
assert train["Gamma_GT"].dropna().gt(0).all()
assert len(sample_submission) == len(test)
assert sample_submission.columns.tolist() == [ID_COLUMN, "Smoking", "Gamma_GT"]

print(f"Train rows/columns: {train.shape}")
print(f"Test rows/columns:  {test.shape}")
print(f"Predictor columns:  {len(FEATURES)}")
print(f"Training targets:   {TARGETS}")
print(f"Test IDs unique:    {test[ID_COLUMN].is_unique}")
print(f"Train IDs unique:   {train[ID_COLUMN].is_unique}")
print("Feature schemas match:", set(FEATURES) == set(test.columns) - {ID_COLUMN})

Train rows/columns: (9002, 41)
Test rows/columns:  (6399, 39)
Predictor columns:  38
Training targets:   ['Smoking', 'Gamma_GT']
Test IDs unique:    True
Train IDs unique:   True
Feature schemas match: True


In [12]:
print("Training data types:")
print(train.dtypes.value_counts().to_string())

print("\nMissing values by column (%), highest first:")
missing_percent = train.isna().mean().mul(100).sort_values(ascending=False)
print(missing_percent[missing_percent.gt(0)].round(1).to_string())

print("\nSmoking target balance:")
print(train["Smoking"].value_counts(dropna=False).sort_index().to_string())
print((train["Smoking"].value_counts(normalize=True).sort_index().mul(100).round(1)).to_string())

print("\nGamma_GT summary (U/L):")
print(train["Gamma_GT"].describe(percentiles=[0.5, 0.9, 0.99]).to_string())

Training data types:
float64    36
int64       3
str         2

Missing values by column (%), highest first:
Exhaled_CO_ppm              23.4
Liver_Echo_Grade            16.8
Alcohol_Units_Week          13.9
Household_Smoke_Exposure    11.2
Sleep_Hours_Avg              9.0
Family_History_CVD           8.1
POC_GGT                      7.9
POC_Batch                    7.9
Urine_Protein                6.1
LDL                          5.0
Urine_Creatinine             4.8
Urine_Cotinine_ng_mL         4.8
Urine_EtG_ng_mL              4.8
Urine_Plate                  4.8
Triglyceride                 4.0
Serum_Creatinine             3.3
AST                          2.9
ALT                          2.9
Eyesight_L                   2.1
Eyesight_R                   1.9

Smoking target balance:
Smoking
0    5731
1    3271
Smoking
0    63.7
1    36.3

Gamma_GT summary (U/L):
count    9002.000000
mean       26.164892
std        19.353244
min         0.998000
50%        20.843000
90%        46.008700

In [13]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 9002 entries, 0 to 9001
Data columns (total 41 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        9002 non-null   int64  
 1   Triglyceride              8639 non-null   float64
 2   Household_Smoke_Exposure  7992 non-null   float64
 3   Waist_cm                  9002 non-null   float64
 4   Visit_Quarter             9002 non-null   str    
 5   Hemoglobin                9002 non-null   float64
 6   AST                       8739 non-null   float64
 7   Cholesterol_Total         9002 non-null   float64
 8   Age                       9002 non-null   float64
 9   LDL                       8551 non-null   float64
 10  Weight_kg                 9002 non-null   float64
 11  ALT                       8742 non-null   float64
 12  Serum_Creatinine          8704 non-null   float64
 13  Urine_Protein             8456 non-null   float64
 14  Fasting_Glucose    

In [14]:
train['Urine_Protein'].value_counts()

Urine_Protein
1.0    8348
2.0      86
3.0      19
4.0       3
Name: count, dtype: int64

## Step 2: Missing-Value Cleaning Decisions

We will not replace unknown values with zero. The policies below keep the meaning of each field and retain signals about which measurements were unavailable:

- **Median + missing indicator:** continuous lab and lifestyle measurements with missing values (`Triglyceride`, `AST`, `ALT`, `Serum_Creatinine`, `Sleep_Hours_Avg`, `Alcohol_Units_Week`, `Exhaled_CO_ppm`, urine measurements, and `POC_GGT`). Median is robust to skew and outliers; the indicator lets a model distinguish an imputed value from a measured one. The imputer will be fitted within each training fold later.
- **Ordinal values:** `Household_Smoke_Exposure` (0–4), `Liver_Echo_Grade` (0–3), and `Urine_Protein` (grades 1–4) keep their numeric ordering; use median imputation plus missing indicators, not one-hot encoding.
- **Binary history:** `Family_History_CVD` uses the most frequent observed value plus a missing indicator. Missing family history is not the same as no history.
- **Derived LDL:** when LDL is absent but total cholesterol, HDL, and triglycerides are observed, estimate LDL using the dictionary formula. Retain an `LDL_Was_Missing` flag because a derived value is not a direct measurement.
- **Eyesight special code:** `9.9` means blindness, not excellent eyesight. Record a separate blindness flag and set the acuity value to missing. Any other missing acuity readings will use median imputation plus a missing indicator.
- **Panel availability:** add a flag when the urine panel is entirely missing. Keep per-analyte missing indicators too.
- **Identifier-like fields:** initially exclude `Urine_Plate` and `POC_Batch` from model features. They identify assay batches, not people; we can compare them later with validation to see whether they add reproducible signal.

This cell performs only deterministic, meaning-based transformations. Statistical imputation is deliberately deferred until the validation split exists to prevent leakage.

In [3]:
import numpy as np


def prepare_features(frame):
    cleaned = frame[FEATURES].copy()

    # 9.9 is a blindness code, so preserve it as a flag, not an acuity value.
    for eye in ["Eyesight_L", "Eyesight_R"]:
        blind_column = f"{eye}_Blind"
        cleaned[blind_column] = cleaned[eye].eq(9.9).astype("int8")
        cleaned.loc[cleaned[eye].eq(9.9), eye] = np.nan

    # Reconstruct missing LDL only when all formula inputs were actually measured.
    ldl_was_missing = cleaned["LDL"].isna()
    lipid_inputs_observed = cleaned[["Cholesterol_Total", "HDL", "Triglyceride"]].notna().all(axis=1)
    calculated_ldl = (
        cleaned["Cholesterol_Total"]
        - cleaned["HDL"]
        - cleaned["Triglyceride"] / 5
    ).clip(lower=0)
    derive_ldl = ldl_was_missing & lipid_inputs_observed
    cleaned.loc[derive_ldl, "LDL"] = calculated_ldl.loc[derive_ldl]
    cleaned["LDL_Was_Missing"] = ldl_was_missing.astype("int8")

    urine_measurements = ["Urine_Creatinine", "Urine_Cotinine_ng_mL", "Urine_EtG_ng_mL"]
    cleaned["Urine_Panel_Missing"] = cleaned[urine_measurements].isna().all(axis=1).astype("int8")

    audit = {
        "blindness_codes_left": int(cleaned["Eyesight_L_Blind"].sum()),
        "blindness_codes_right": int(cleaned["Eyesight_R_Blind"].sum()),
        "LDL_values_derived": int(derive_ldl.sum()),
        "urine_panels_missing": int(cleaned["Urine_Panel_Missing"].sum()),
    }
    return cleaned, audit


train_features, train_cleaning_audit = prepare_features(train)
test_features, test_cleaning_audit = prepare_features(test)

# Batch/plate identifiers are held out of the first baseline and can be tested later.
MODEL_FEATURES = [column for column in train_features.columns if column not in ["Urine_Plate", "POC_Batch"]]

assert train_features.columns.tolist() == test_features.columns.tolist()
assert "Smoking" not in train_features.columns and "Gamma_GT" not in train_features.columns
assert not train_features["Eyesight_L"].eq(9.9).any()
assert not train_features["Eyesight_R"].eq(9.9).any()

print("Deterministic cleaning audit (train):", train_cleaning_audit)
print("Deterministic cleaning audit (test): ", test_cleaning_audit)
print("Predictor columns before excluding batch IDs:", train_features.shape[1])
print("Baseline model predictor columns:", len(MODEL_FEATURES))
print("Missing values left for fold-fitted imputation:", int(train_features[MODEL_FEATURES].isna().sum().sum()))

Deterministic cleaning audit (train): {'blindness_codes_left': 0, 'blindness_codes_right': 0, 'LDL_values_derived': 430, 'urine_panels_missing': 430}
Deterministic cleaning audit (test):  {'blindness_codes_left': 0, 'blindness_codes_right': 0, 'LDL_values_derived': 318, 'urine_panels_missing': 332}
Predictor columns before excluding batch IDs: 42
Baseline model predictor columns: 40
Missing values left for fold-fitted imputation: 11522


### Step 2.1: Impute the Training Copy

This step creates `train_cleaned` without modifying the raw `train` dataframe. Before filling values, it saves a missingness flag for every predictor, so the models can distinguish measured values from imputed ones.

- Continuous and ordinal numeric fields use the **training median**, which is robust to skew and outliers and preserves numeric ordering.
- `Family_History_CVD` uses its **most frequent observed value**, because it is a binary history field rather than a continuous measurement.
- `Visit_Quarter` and `Screening_Center` use their **most frequent observed category**.
- `Workplace_ID`, `Urine_Plate`, and `POC_Batch` are treated as categorical identifiers; missing identifiers are labeled `Missing`, not replaced with a fake numeric ID. Plate and batch remain excluded from the initial model feature list.
- Imputation values are learned from `train_features` and reused for test transformation. This full-training copy is for inspection/final refit only; for validation, these same rules must be fit inside each training fold to avoid leakage.

In [16]:
CATEGORICAL_COLUMNS = ["Visit_Quarter", "Screening_Center"]
IDENTIFIER_COLUMNS = ["Workplace_ID", "Urine_Plate", "POC_Batch"]
BINARY_COLUMNS = ["Family_History_CVD"]

# Learn imputation values from train only, then apply the same values to both copies.
missing_columns = [column for column in train_features.columns if train_features[column].isna().any()]
train_imputed = train_features.copy()
test_imputed = test_features.copy()
imputation_values = {}

for column in train_features.columns:
    if column in IDENTIFIER_COLUMNS:
        train_imputed[column] = train_imputed[column].astype("string").fillna("Missing")
        test_imputed[column] = test_imputed[column].astype("string").fillna("Missing")
    elif column in CATEGORICAL_COLUMNS:
        fill_value = train_features[column].mode(dropna=True).iloc[0]
        imputation_values[column] = fill_value
        train_imputed[column] = train_imputed[column].fillna(fill_value)
        test_imputed[column] = test_imputed[column].fillna(fill_value)
    elif column in BINARY_COLUMNS:
        fill_value = train_features[column].mode(dropna=True).iloc[0]
        imputation_values[column] = fill_value
        train_imputed[column] = train_imputed[column].fillna(fill_value)
        test_imputed[column] = test_imputed[column].fillna(fill_value)
    elif train_features[column].isna().any():
        fill_value = train_features[column].median()
        if pd.isna(fill_value):
            raise ValueError(f"Cannot impute {column}: it has no observed training values.")
        imputation_values[column] = fill_value
        train_imputed[column] = train_imputed[column].fillna(fill_value)
        test_imputed[column] = test_imputed[column].fillna(fill_value)

for column in missing_columns:
    indicator = f"{column}_Was_Missing"
    train_imputed[indicator] = train_features[column].isna().astype("int8")
    test_imputed[indicator] = test_features[column].isna().astype("int8")

train_cleaned = pd.concat([train[[ID_COLUMN]], train_imputed, train[TARGETS]], axis=1)
test_cleaned = pd.concat([test[[ID_COLUMN]], test_imputed], axis=1)

# Exclude assay IDs and their missingness flags from the initial model baseline.
EXCLUDED_BASELINE_COLUMNS = [
    "Urine_Plate", "POC_Batch", "Urine_Plate_Was_Missing", "POC_Batch_Was_Missing"
]
CLEAN_MODEL_FEATURES = [
    column for column in train_imputed.columns if column not in EXCLUDED_BASELINE_COLUMNS
]

assert train_cleaned.isna().sum().sum() == 0
assert test_cleaned.isna().sum().sum() == 0
assert train_cleaned.columns.tolist() == [ID_COLUMN] + train_imputed.columns.tolist() + TARGETS
assert test_cleaned.columns.tolist() == [ID_COLUMN] + test_imputed.columns.tolist()
assert train_cleaned["Smoking"].equals(train["Smoking"])
assert train_cleaned["Gamma_GT"].equals(train["Gamma_GT"])

print("Cleaned training shape:", train_cleaned.shape)
print("Cleaned test feature shape:", test_cleaned.shape)
print("Training predictors with learned imputation values:", len(imputation_values))
print("Missingness indicators added:", len(missing_columns))
print("Remaining missing cells in train:", int(train_cleaned.isna().sum().sum()))
print("Remaining missing cells in test:", int(test_cleaned.isna().sum().sum()))
print("Model feature count after excluding assay IDs:", len(CLEAN_MODEL_FEATURES))
print("Example learned imputation values:")
print(pd.Series(imputation_values).head(12).to_string())

Cleaned training shape: (9002, 64)
Cleaned test feature shape: (6399, 62)
Training predictors with learned imputation values: 20
Missingness indicators added: 20
Remaining missing cells in train: 0
Remaining missing cells in test: 0
Model feature count after excluding assay IDs: 57
Example learned imputation values:
Triglyceride                   89.9
Household_Smoke_Exposure        1.0
Visit_Quarter                    Q2
AST                          21.029
LDL                         112.847
ALT                          18.019
Serum_Creatinine              0.896
Urine_Protein                   1.0
Eyesight_R                    1.001
Sleep_Hours_Avg                6.87
Alcohol_Units_Week           6.6065
Liver_Echo_Grade                1.0


In [17]:
train_cleaned.info()

<class 'pandas.DataFrame'>
RangeIndex: 9002 entries, 0 to 9001
Data columns (total 64 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   ID                                    9002 non-null   int64  
 1   Triglyceride                          9002 non-null   float64
 2   Household_Smoke_Exposure              9002 non-null   float64
 3   Waist_cm                              9002 non-null   float64
 4   Visit_Quarter                         9002 non-null   str    
 5   Hemoglobin                            9002 non-null   float64
 6   AST                                   9002 non-null   float64
 7   Cholesterol_Total                     9002 non-null   float64
 8   Age                                   9002 non-null   float64
 9   LDL                                   9002 non-null   float64
 10  Weight_kg                             9002 non-null   float64
 11  ALT                         

## Step 3: Scale Continuous Predictors

Use `RobustScaler` for continuous measurements only. It subtracts each training column's median and divides by its interquartile range, making feature scales comparable while reducing sensitivity to extreme values. The scaler is fitted on the cleaned training predictors and reused unchanged for test predictors.

Keep ordinal codes (`Household_Smoke_Exposure`, `Liver_Echo_Grade`, `Urine_Protein`, hearing values), binary/missingness flags, categories, and identifier-like columns unscaled so their codes and meanings remain intact. Do not scale `ID`, `Smoking`, or `Gamma_GT`. This creates scaled copies for scale-sensitive models; tree-based models generally do not need feature scaling. In validation, fit a fresh scaler within each training fold.

In [18]:
from sklearn.preprocessing import RobustScaler

CONTINUOUS_COLUMNS = [
    "Age", "Height_cm", "Weight_kg", "Waist_cm", "BMI",
    "BP_Systolic", "BP_Diastolic", "Fasting_Glucose", "Cholesterol_Total",
    "Triglyceride", "HDL", "LDL", "Hemoglobin", "Serum_Creatinine",
    "AST", "ALT", "Eyesight_L", "Eyesight_R", "Sleep_Hours_Avg",
    "Alcohol_Units_Week", "Exhaled_CO_ppm", "Metabolic_Age_Gap",
    "Urine_Creatinine", "Urine_Cotinine_ng_mL", "Urine_EtG_ng_mL", "POC_GGT",
]
SCALABLE_COLUMNS = [
    column for column in CONTINUOUS_COLUMNS
    if column in CLEAN_MODEL_FEATURES
]

# Fit only on cleaned training predictors; reuse exactly the same transform on test.
scaler = RobustScaler()
scaler.fit(train_cleaned[SCALABLE_COLUMNS])

train_scaled = train_cleaned.copy()
test_scaled = test_cleaned.copy()
train_scaled.loc[:, SCALABLE_COLUMNS] = scaler.transform(train_cleaned[SCALABLE_COLUMNS])
test_scaled.loc[:, SCALABLE_COLUMNS] = scaler.transform(test_cleaned[SCALABLE_COLUMNS])

# Targets, IDs, categorical/ordinal values, and binary missingness flags are untouched.
assert train_scaled[ID_COLUMN].equals(train_cleaned[ID_COLUMN])
assert test_scaled[ID_COLUMN].equals(test_cleaned[ID_COLUMN])
assert train_scaled[TARGETS].equals(train_cleaned[TARGETS])
assert train_scaled.isna().sum().sum() == 0
assert test_scaled.isna().sum().sum() == 0
assert train_scaled.columns.tolist() == [ID_COLUMN] + train_imputed.columns.tolist() + TARGETS
assert test_scaled.columns.tolist() == [ID_COLUMN] + test_imputed.columns.tolist()

print("Scaled continuous columns:", len(SCALABLE_COLUMNS))
print("Scaled columns:", SCALABLE_COLUMNS)
print("Training medians after scaling (first 8):")
print(train_scaled[SCALABLE_COLUMNS].median().head(8).round(3).to_string())
print("Training IQRs after scaling (first 8):")
print((train_scaled[SCALABLE_COLUMNS].quantile(0.75) - train_scaled[SCALABLE_COLUMNS].quantile(0.25)).head(8).round(3).to_string())
print("Remaining missing cells - train:", int(train_scaled.isna().sum().sum()))
print("Remaining missing cells - test:", int(test_scaled.isna().sum().sum()))
print("Target columns unchanged:", train_scaled[TARGETS].equals(train_cleaned[TARGETS]))

Scaled continuous columns: 26
Scaled columns: ['Age', 'Height_cm', 'Weight_kg', 'Waist_cm', 'BMI', 'BP_Systolic', 'BP_Diastolic', 'Fasting_Glucose', 'Cholesterol_Total', 'Triglyceride', 'HDL', 'LDL', 'Hemoglobin', 'Serum_Creatinine', 'AST', 'ALT', 'Eyesight_L', 'Eyesight_R', 'Sleep_Hours_Avg', 'Alcohol_Units_Week', 'Exhaled_CO_ppm', 'Metabolic_Age_Gap', 'Urine_Creatinine', 'Urine_Cotinine_ng_mL', 'Urine_EtG_ng_mL', 'POC_GGT']
Training medians after scaling (first 8):
Age                0.0
Height_cm          0.0
Weight_kg          0.0
Waist_cm           0.0
BMI                0.0
BP_Systolic       -0.0
BP_Diastolic       0.0
Fasting_Glucose    0.0
Training IQRs after scaling (first 8):
Age                1.0
Height_cm          1.0
Weight_kg          1.0
Waist_cm           1.0
BMI                1.0
BP_Systolic        1.0
BP_Diastolic       1.0
Fasting_Glucose    1.0
Remaining missing cells - train: 0
Remaining missing cells - test: 0
Target columns unchanged: True


## Step 4: First Validation Model Run

Start with one CatBoost model per target. CatBoost is a boosted-tree method suited to tabular data; it supports categorical predictors and can learn numeric missing-value routing. We use the original, unscaled, meaning-cleaned predictors because tree models do not need scaling.

- `Smoking`: `CatBoostClassifier` with Logloss training objective and AUC evaluation. The holdout split is stratified because the target is binary. We evaluate the probability for class 1, not hard labels.
- `Gamma_GT`: `CatBoostRegressor` trained on `log1p(Gamma_GT)`. Convert predictions back with `expm1`, clip to the competition range `[1, 1000]`, and evaluate RMSLE.

Only categorical missing values are replaced with the explicit category `Missing`; this is a deterministic representation rather than a learned statistic. Numeric NaNs are left for CatBoost's native handling. This is an initial validation run, not final tuning. The same fixed random seed is used for repeatability.

In [4]:
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import mean_squared_log_error, roc_auc_score
from sklearn.model_selection import train_test_split

VALIDATION_SIZE = 0.2
RANDOM_STATE = 42
EXCLUDED_MODEL_COLUMNS = ["Urine_Plate", "POC_Batch"]
CATBOOST_CATEGORICAL_COLUMNS = ["Visit_Quarter", "Screening_Center", "Workplace_ID"]
MODEL_INPUT_COLUMNS = [
    column for column in train_features.columns
    if column not in EXCLUDED_MODEL_COLUMNS
]


def model_ready_features(frame):
    ready = frame[MODEL_INPUT_COLUMNS].copy()
    for column in CATBOOST_CATEGORICAL_COLUMNS:
        ready[column] = ready[column].astype("string").fillna("Missing").astype(str)
    return ready


X_all = model_ready_features(train_features)
X_train, X_valid, y_smoke_train, y_smoke_valid = train_test_split(
    X_all,
    train["Smoking"],
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=train["Smoking"],
)

smoking_model = CatBoostClassifier(
    iterations=1200,
    depth=6,
    learning_rate=0.04,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
smoking_model.fit(
    X_train,
    y_smoke_train,
    cat_features=CATBOOST_CATEGORICAL_COLUMNS,
    eval_set=(X_valid, y_smoke_valid),
    early_stopping_rounds=100,
    verbose=False,
)
smoking_valid_probability = smoking_model.predict_proba(X_valid)[:, 1]
smoking_valid_auc = roc_auc_score(y_smoke_valid, smoking_valid_probability)

X_train_gamma, X_valid_gamma, y_gamma_train, y_gamma_valid = train_test_split(
    X_all,
    train["Gamma_GT"],
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)
gamma_model = CatBoostRegressor(
    iterations=1200,
    depth=6,
    learning_rate=0.04,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
gamma_model.fit(
    X_train_gamma,
    np.log1p(y_gamma_train),
    cat_features=CATBOOST_CATEGORICAL_COLUMNS,
    eval_set=(X_valid_gamma, np.log1p(y_gamma_valid)),
    early_stopping_rounds=100,
    verbose=False,
)
gamma_valid_log_prediction = gamma_model.predict(X_valid_gamma)
gamma_valid_prediction = np.expm1(gamma_valid_log_prediction).clip(1, 1000)
gamma_valid_rmsle = mean_squared_log_error(
    y_gamma_valid, gamma_valid_prediction
) ** 0.5

competition_score = 0.5 * (2 * smoking_valid_auc - 1) + 0.5 * (1 - gamma_valid_rmsle / 0.5041)
validation_results = {
    "smoking_auc": smoking_valid_auc,
    "gamma_gt_rmsle": gamma_valid_rmsle,
    "competition_score": competition_score,
    "smoking_best_iteration": smoking_model.get_best_iteration(),
    "gamma_gt_best_iteration": gamma_model.get_best_iteration(),
}

print(f"Smoking validation AUC: {smoking_valid_auc:.5f}")
print(f"Gamma_GT validation RMSLE: {gamma_valid_rmsle:.5f}")
print(f"Combined validation score: {competition_score:.5f}")
print("Smoking best iteration:", smoking_model.get_best_iteration())
print("Gamma_GT best iteration:", gamma_model.get_best_iteration())
print("Validation rows per task:", len(y_smoke_valid), len(y_gamma_valid))

Smoking validation AUC: 0.94251
Gamma_GT validation RMSLE: 0.28141
Combined validation score: 0.66339
Smoking best iteration: 562
Gamma_GT best iteration: 714
Validation rows per task: 1801 1801


## Step 5: Compare XGBoost on the Same Holdouts

XGBoost is the second boosted-tree candidate. Train a classifier and a regressor separately, using the exact same task-specific validation rows as the CatBoost run. Use XGBoost's histogram tree method and native categorical support; numeric missing values remain NaN. The classifier is evaluated by AUC, while the regressor learns `log1p(Gamma_GT)` and is evaluated by competition RMSLE after inverse-transforming and clipping predictions.

In [5]:
from xgboost import XGBClassifier, XGBRegressor

# Build category dtypes once before splitting so train/validation category codes align.
X_xgb_all = model_ready_features(train_features)
for column in CATBOOST_CATEGORICAL_COLUMNS:
    X_xgb_all[column] = X_xgb_all[column].astype("category")

X_xgb_train_smoke, X_xgb_valid_smoke, y_xgb_smoke_train, y_xgb_smoke_valid = train_test_split(
    X_xgb_all,
    train["Smoking"],
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
    stratify=train["Smoking"],
)

xgb_smoking_model = XGBClassifier(
    n_estimators=1500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_smoking_model.fit(
    X_xgb_train_smoke,
    y_xgb_smoke_train,
    eval_set=[(X_xgb_valid_smoke, y_xgb_smoke_valid)],
    verbose=False,
)
xgb_smoking_probability = xgb_smoking_model.predict_proba(X_xgb_valid_smoke)[:, 1]
xgb_smoking_auc = roc_auc_score(y_xgb_smoke_valid, xgb_smoking_probability)

X_xgb_train_gamma, X_xgb_valid_gamma, y_xgb_gamma_train, y_xgb_gamma_valid = train_test_split(
    X_xgb_all,
    train["Gamma_GT"],
    test_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)
xgb_gamma_model = XGBRegressor(
    n_estimators=1500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_gamma_model.fit(
    X_xgb_train_gamma,
    np.log1p(y_xgb_gamma_train),
    eval_set=[(X_xgb_valid_gamma, np.log1p(y_xgb_gamma_valid))],
    verbose=False,
)
xgb_gamma_log_prediction = xgb_gamma_model.predict(X_xgb_valid_gamma)
xgb_gamma_prediction = np.expm1(xgb_gamma_log_prediction).clip(1, 1000)
xgb_gamma_rmsle = mean_squared_log_error(y_xgb_gamma_valid, xgb_gamma_prediction) ** 0.5
xgb_combined_score = 0.5 * (2 * xgb_smoking_auc - 1) + 0.5 * (1 - xgb_gamma_rmsle / 0.5041)

# Confirm this comparison used the exact same row indices as each CatBoost task.
assert X_xgb_train_smoke.index.equals(X_train.index)
assert X_xgb_valid_smoke.index.equals(X_valid.index)
assert X_xgb_train_gamma.index.equals(X_train_gamma.index)
assert X_xgb_valid_gamma.index.equals(X_valid_gamma.index)

model_comparison = pd.DataFrame([
    {
        "Model": "CatBoost",
        "Smoking_AUC": smoking_valid_auc,
        "Gamma_GT_RMSLE": gamma_valid_rmsle,
        "Combined_Score": competition_score,
    },
    {
        "Model": "XGBoost",
        "Smoking_AUC": xgb_smoking_auc,
        "Gamma_GT_RMSLE": xgb_gamma_rmsle,
        "Combined_Score": xgb_combined_score,
    },
]).set_index("Model")

print(model_comparison.round(5).to_string())
print("\nXGBoost best iteration (Smoking):", xgb_smoking_model.best_iteration)
print("XGBoost best iteration (Gamma_GT):", xgb_gamma_model.best_iteration)
print("\nExact task-specific CatBoost holdout rows reused:", True)

          Smoking_AUC  Gamma_GT_RMSLE  Combined_Score
Model                                                
CatBoost      0.94251         0.28141         0.66339
XGBoost       0.93160         0.30285         0.63121

XGBoost best iteration (Smoking): 609
XGBoost best iteration (Gamma_GT): 420

Exact task-specific CatBoost holdout rows reused: True


## Step 6: Refit on All Training Rows and Predict Test

CatBoost led the initial validation comparison for both targets, so use it for this first full-data run. Refit a classifier on every labeled `Smoking` row and a separate regressor on `log1p(Gamma_GT)` for every labeled Gamma-GT row. The test file contains no targets, so this step produces predictions for submission rather than test-set accuracy. Final Gamma-GT predictions are inverted with `expm1` and clipped to `[1, 1000]`.

In [7]:
FINAL_SMOKING_ITERATIONS = smoking_model.get_best_iteration() + 1
FINAL_GAMMA_ITERATIONS = gamma_model.get_best_iteration() + 1

final_train_features = model_ready_features(train_features)
final_test_features = model_ready_features(test_features)
assert final_train_features.columns.tolist() == final_test_features.columns.tolist()

final_smoking_model = CatBoostClassifier(
    iterations=FINAL_SMOKING_ITERATIONS,
    depth=6,
    learning_rate=0.04,
    loss_function="Logloss",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
final_smoking_model.fit(
    final_train_features,
    train["Smoking"],
    cat_features=CATBOOST_CATEGORICAL_COLUMNS,
    verbose=False,
)

final_gamma_model = CatBoostRegressor(
    iterations=FINAL_GAMMA_ITERATIONS,
    depth=6,
    learning_rate=0.04,
    loss_function="RMSE",
    random_seed=RANDOM_STATE,
    verbose=False,
    allow_writing_files=False,
)
final_gamma_model.fit(
    final_train_features,
    np.log1p(train["Gamma_GT"]),
    cat_features=CATBOOST_CATEGORICAL_COLUMNS,
    verbose=False,
)

submission = sample_submission.copy()
submission[ID_COLUMN] = test[ID_COLUMN].to_numpy()
submission["Smoking"] = final_smoking_model.predict_proba(final_test_features)[:, 1]
submission["Gamma_GT"] = np.expm1(final_gamma_model.predict(final_test_features)).clip(1, 1000)

assert submission.shape == (len(test), 3)
assert submission.columns.tolist() == [ID_COLUMN, "Smoking", "Gamma_GT"]
assert submission[ID_COLUMN].to_numpy().tolist() == test[ID_COLUMN].to_numpy().tolist()
assert submission[["Smoking", "Gamma_GT"]].notna().all().all()
assert np.isfinite(submission[["Smoking", "Gamma_GT"]].to_numpy()).all()
assert submission["Smoking"].between(0, 1).all()
assert submission["Smoking"].nunique() > 1
assert submission["Gamma_GT"].between(1, 1000).all()

submission_path = DATA_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)
verified_submission = pd.read_csv(submission_path)
assert verified_submission.columns.tolist() == [ID_COLUMN, "Smoking", "Gamma_GT"]
assert verified_submission.shape == (len(test), 3)
assert verified_submission[ID_COLUMN].to_numpy().tolist() == test[ID_COLUMN].to_numpy().tolist()
assert np.isfinite(verified_submission[["Smoking", "Gamma_GT"]].to_numpy()).all()
assert np.allclose(
    verified_submission[["Smoking", "Gamma_GT"]].to_numpy(),
    submission[["Smoking", "Gamma_GT"]].to_numpy(),
    rtol=1e-12,
    atol=1e-12,
)

print("Final model rows used:", len(train))
print("Final smoking iterations:", FINAL_SMOKING_ITERATIONS)
print("Final Gamma_GT iterations:", FINAL_GAMMA_ITERATIONS)
print("Submission shape:", verified_submission.shape)
print("Submission path:", submission_path)
print("Smoking prediction range:", tuple(verified_submission["Smoking"].agg(["min", "max"]).round(5)))
print("Gamma_GT prediction range:", tuple(verified_submission["Gamma_GT"].agg(["min", "max"]).round(3)))
print(verified_submission.head().to_string(index=False))

Final model rows used: 9002
Final smoking iterations: 563
Final Gamma_GT iterations: 715
Submission shape: (6399, 3)
Submission path: c:\Users\يثمم\Desktop\AI_Hackathon\dsc-modeling-hackathon-ml-arena\submission.csv
Smoking prediction range: (0.0001, 0.99905)
Gamma_GT prediction range: (8.813, 170.894)
  ID  Smoking  Gamma_GT
9002 0.346252 13.817254
9003 0.001851 15.654366
9004 0.037853 32.113774
9005 0.966709 22.793980
9006 0.008023 18.320338


## Step 7: Five-Fold Stability Check

Compare CatBoost and XGBoost across five folds. `Smoking` uses stratified folds; `Gamma_GT` uses shuffled K-folds. Each model receives out-of-fold predictions for every training row, and we report pooled OOF metrics plus the mean and standard deviation across folds.

To keep this a focused stability check, model settings and iteration counts are fixed from the first holdout run; there is no early stopping inside these folds. Therefore these CV results help assess consistency and compare candidates, but are not a perfectly untouched estimate because iteration counts were previously selected using a holdout from this same training set. If we tune extensively, reserve a separate final evaluation or use nested CV.

In [8]:
from sklearn.model_selection import KFold, StratifiedKFold

smoking_target = train["Smoking"].to_numpy()
gamma_target = train["Gamma_GT"].to_numpy()
X_cv_catboost = model_ready_features(train_features)
X_cv_xgboost = X_cv_catboost.copy()
for column in CATBOOST_CATEGORICAL_COLUMNS:
    X_cv_xgboost[column] = X_cv_xgboost[column].astype("category")

catboost_smoking_oof = np.zeros(len(train))
xgboost_smoking_oof = np.zeros(len(train))
catboost_gamma_oof = np.zeros(len(train))
xgboost_gamma_oof = np.zeros(len(train))
smoking_fold_auc = {"CatBoost": [], "XGBoost": []}
gamma_fold_rmsle = {"CatBoost": [], "XGBoost": []}

smoking_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold_number, (fit_indices, valid_indices) in enumerate(
    smoking_cv.split(X_cv_catboost, smoking_target), start=1
):
    cat_classifier = CatBoostClassifier(
        iterations=FINAL_SMOKING_ITERATIONS,
        depth=6,
        learning_rate=0.04,
        loss_function="Logloss",
        random_seed=RANDOM_STATE + fold_number,
        verbose=False,
        allow_writing_files=False,
    )
    cat_classifier.fit(
        X_cv_catboost.iloc[fit_indices],
        smoking_target[fit_indices],
        cat_features=CATBOOST_CATEGORICAL_COLUMNS,
        verbose=False,
    )
    cat_probability = cat_classifier.predict_proba(X_cv_catboost.iloc[valid_indices])[:, 1]
    catboost_smoking_oof[valid_indices] = cat_probability
    smoking_fold_auc["CatBoost"].append(
        roc_auc_score(smoking_target[valid_indices], cat_probability)
    )

    xgb_classifier = XGBClassifier(
        n_estimators=int(xgb_smoking_model.best_iteration) + 1,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        enable_categorical=True,
        random_state=RANDOM_STATE + fold_number,
        n_jobs=-1,
    )
    xgb_classifier.fit(
        X_cv_xgboost.iloc[fit_indices],
        smoking_target[fit_indices],
        verbose=False,
    )
    xgb_probability = xgb_classifier.predict_proba(X_cv_xgboost.iloc[valid_indices])[:, 1]
    xgboost_smoking_oof[valid_indices] = xgb_probability
    smoking_fold_auc["XGBoost"].append(
        roc_auc_score(smoking_target[valid_indices], xgb_probability)
    )
    print(f"Completed smoking fold {fold_number}/5")

gamma_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for fold_number, (fit_indices, valid_indices) in enumerate(
    gamma_cv.split(X_cv_catboost), start=1
):
    cat_regressor = CatBoostRegressor(
        iterations=FINAL_GAMMA_ITERATIONS,
        depth=6,
        learning_rate=0.04,
        loss_function="RMSE",
        random_seed=RANDOM_STATE + fold_number,
        verbose=False,
        allow_writing_files=False,
    )
    cat_regressor.fit(
        X_cv_catboost.iloc[fit_indices],
        np.log1p(gamma_target[fit_indices]),
        cat_features=CATBOOST_CATEGORICAL_COLUMNS,
        verbose=False,
    )
    cat_prediction = np.expm1(cat_regressor.predict(X_cv_catboost.iloc[valid_indices])).clip(1, 1000)
    catboost_gamma_oof[valid_indices] = cat_prediction
    gamma_fold_rmsle["CatBoost"].append(
        mean_squared_log_error(gamma_target[valid_indices], cat_prediction) ** 0.5
    )

    xgb_regressor = XGBRegressor(
        n_estimators=int(xgb_gamma_model.best_iteration) + 1,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        eval_metric="rmse",
        tree_method="hist",
        enable_categorical=True,
        random_state=RANDOM_STATE + fold_number,
        n_jobs=-1,
    )
    xgb_regressor.fit(
        X_cv_xgboost.iloc[fit_indices],
        np.log1p(gamma_target[fit_indices]),
        verbose=False,
    )
    xgb_prediction = np.expm1(xgb_regressor.predict(X_cv_xgboost.iloc[valid_indices])).clip(1, 1000)
    xgboost_gamma_oof[valid_indices] = xgb_prediction
    gamma_fold_rmsle["XGBoost"].append(
        mean_squared_log_error(gamma_target[valid_indices], xgb_prediction) ** 0.5
    )
    print(f"Completed Gamma_GT fold {fold_number}/5")

cv_rows = []
for model_name, smoking_oof, gamma_oof in [
    ("CatBoost", catboost_smoking_oof, catboost_gamma_oof),
    ("XGBoost", xgboost_smoking_oof, xgboost_gamma_oof),
]:
    pooled_auc = roc_auc_score(smoking_target, smoking_oof)
    pooled_rmsle = mean_squared_log_error(gamma_target, gamma_oof) ** 0.5
    pooled_score = 0.5 * (2 * pooled_auc - 1) + 0.5 * (1 - pooled_rmsle / 0.5041)
    cv_rows.append({
        "Model": model_name,
        "Smoking_AUC_mean": np.mean(smoking_fold_auc[model_name]),
        "Smoking_AUC_std": np.std(smoking_fold_auc[model_name], ddof=1),
        "Smoking_AUC_OOF": pooled_auc,
        "Gamma_RMSLE_mean": np.mean(gamma_fold_rmsle[model_name]),
        "Gamma_RMSLE_std": np.std(gamma_fold_rmsle[model_name], ddof=1),
        "Gamma_RMSLE_OOF": pooled_rmsle,
        "Combined_OOF_score": pooled_score,
    })

cv_comparison = pd.DataFrame(cv_rows).set_index("Model")
assert np.isfinite(cv_comparison.to_numpy()).all()
print("\nFive-fold metrics (mean/std per-fold and pooled out-of-fold):")
print(cv_comparison.round(5).to_string())
print("\nSmoking fold AUCs:", {key: np.round(value, 5).tolist() for key, value in smoking_fold_auc.items()})
print("Gamma_GT fold RMSLEs:", {key: np.round(value, 5).tolist() for key, value in gamma_fold_rmsle.items()})

Completed smoking fold 1/5
Completed smoking fold 2/5
Completed smoking fold 3/5
Completed smoking fold 4/5
Completed smoking fold 5/5
Completed Gamma_GT fold 1/5
Completed Gamma_GT fold 2/5
Completed Gamma_GT fold 3/5
Completed Gamma_GT fold 4/5
Completed Gamma_GT fold 5/5

Five-fold metrics (mean/std per-fold and pooled out-of-fold):
          Smoking_AUC_mean  Smoking_AUC_std  Smoking_AUC_OOF  Gamma_RMSLE_mean  Gamma_RMSLE_std  Gamma_RMSLE_OOF  Combined_OOF_score
Model                                                                                                                               
CatBoost           0.94673          0.00271          0.94672           0.28548          0.00587          0.28553             0.66351
XGBoost            0.93596          0.00351          0.93595           0.30594          0.00411          0.30596             0.63248

Smoking fold AUCs: {'CatBoost': [0.94896, 0.94664, 0.94509, 0.94976, 0.94318], 'XGBoost': [0.94024, 0.93591, 0.93255, 0.93863, 0.

## Step 8: Targeted Tuning and Feature Comparison

Keep the saved CatBoost/XGBoost five-fold OOF results as the baseline. Compare two predeclared CatBoost configurations per target and tuned XGBoost against CatBoost on identical seeded folds, adding a small set of lab ratios and source-missingness flags. Select the classifier and regressor independently using AUC and RMSLE, refit the chosen pair on all labeled rows, and validate the resulting submission. This is an exploratory selection run on the same folds, not an untouched final estimate; no test-label score is available.

In [2]:
%run model_tuning.py

Smoking: CatBoost fold 1/5 (params['depth']=6)
Smoking: CatBoost fold 2/5 (params['depth']=6)
Smoking: CatBoost fold 3/5 (params['depth']=6)
Smoking: CatBoost fold 4/5 (params['depth']=6)
Smoking: CatBoost fold 5/5 (params['depth']=6)
Smoking: CatBoost fold 1/5 (params['depth']=5)
Smoking: CatBoost fold 2/5 (params['depth']=5)
Smoking: CatBoost fold 3/5 (params['depth']=5)
Smoking: CatBoost fold 4/5 (params['depth']=5)
Smoking: CatBoost fold 5/5 (params['depth']=5)
Smoking: CatBoost fold 1/5 (params['depth']=7)
Smoking: CatBoost fold 2/5 (params['depth']=7)
Smoking: CatBoost fold 3/5 (params['depth']=7)
Smoking: CatBoost fold 4/5 (params['depth']=7)
Smoking: CatBoost fold 5/5 (params['depth']=7)
Smoking: tuned XGBoost fold 1/5
Smoking: tuned XGBoost fold 2/5
Smoking: tuned XGBoost fold 3/5
Smoking: tuned XGBoost fold 4/5
Smoking: tuned XGBoost fold 5/5
Gamma_GT: CatBoost fold 1/5 (params['depth']=6)
Gamma_GT: CatBoost fold 2/5 (params['depth']=6)
Gamma_GT: CatBoost fold 3/5 (params['de

Optimized tool selectionI’m preparing a concise summary in markdown form for the notebook, based on the work we’ve just completed.

Retrieving Notebook summary.

Read , lines 1 to 220

Read , lines 221 to 720

I’ve mapped the notebook workflow and am adding a high-level recap with the recommended next actions for the project.

Edit 

## ✅ Update

I added a markdown summary cell to the notebook with the project recap and recommended next steps.

```markdown
## Project Summary and Next Steps

### What we completed
- Confirmed the competition structure: two targets (`Smoking` and `Gamma_GT`), `ID` as identifier, and no target leakage into features.
- Validated the train/test schema and submission template before modeling.
- Built a meaning-based feature cleaning pipeline:
  - handled blindness flags for `Eyesight_L`/`Eyesight_R`
  - reconstructed missing `LDL` where allowed
  - added missingness indicators and panel-level flags
  - excluded assay identifiers (`Urine_Plate`, `POC_Batch`) from the baseline model features
- Created train-only imputation rules and preserved missingness as signal rather than zero-filling.
- Applied `RobustScaler` only to continuous numeric features for models that need scaled inputs.
- Trained separate baseline models for each task:
  - `CatBoostClassifier` for `Smoking`
  - `CatBoostRegressor` for `Gamma_GT` using `log1p(Gamma_GT)`
- Compared CatBoost vs. XGBoost on the same validation split and found CatBoost to be the stronger initial baseline.
- Refit the selected CatBoost models on the full training data and generated a competition-ready `submission.csv`.
- Ran a 5-fold stability check to compare pooled out-of-fold performance and assess consistency across folds.

### Key takeaways
- The modeling workflow was built around leakage-safe preprocessing.
- The target `Gamma_GT` was handled with log-transform regression to respect the positive, skewed distribution.
- CatBoost was selected as the current leader for both tasks, with XGBoost kept as a strong comparison baseline.
- The submission file is generated and ready to be updated whenever model changes are made.

### Recommended next steps
1. Review the 5-fold results and decide whether CatBoost remains the final choice.
2. Tune CatBoost hyperparameters more deliberately for both targets.
3. Test targeted feature engineering, especially around lab-derived features and missingness signals.
4. Compare a few additional models (for example LightGBM or a tuned XGBoost variant) on the same folds.
5. Rebuild `submission.csv` after any final model changes and keep a clean record of the best-performing run.
6. If the leaderboard score is weak, focus on feature quality and validation discipline before adding too much complexity.

> Current status: the project is in the improvement/tuning stage, with a validated baseline and a generated submission pipeline already in place.
```



Made changes.